# Akkadian to English - Dual ByT5 Ensemble + MBR Decoding

Load pre-trained ByT5 checkpoints and generate predictions using:
1. **Dual-model ensemble** — `byt5-akkadian-optimized-34x` (Model A) + `byt5-akkadian-mbr` (Model B)
2. **Beam + Sampling candidates** — diverse candidate pool per input
3. **MBR (Minimum Bayes Risk) consensus** — chrF++ + proper noun fidelity scoring
4. **Domain-specific post-processing** — fix common Akkadian translation artifacts

**No internet required.** Attach the two ByT5 model datasets.

### Huge thanks to @mattiaangeli and @assiaben for their pre-trained models

In [ ]:
import gc
import math
import os
import re
import random
from contextlib import nullcontext
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


# ==============================================================================
# BF16 utility helpers (from baseline)
# ==============================================================================
def _cuda_bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except Exception:
        return False


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda" and _cuda_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


# ==============================================================================
# 1. CONFIGURATION — Dual ByT5 ensemble (fixed model paths)
# ==============================================================================
@dataclass
class Config:
    test_data_path: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    lexicon_csv_path: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
    output_dir: str = "/kaggle/working/"

    # Fixed model paths — pre-trained ByT5 models
    model_a_path: str = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    model_b_path: str = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6"

    # Sequence lengths
    max_input_length: int = 512
    max_new_tokens: int = 384
    batch_size: int = 2       # ByT5 is larger, use smaller batch
    num_workers: int = 2
    num_buckets: int = 6

    # Candidate generation (aligned with baseline)
    num_beam_cands: int = 4
    num_sample_cands: int = 2
    num_beams: int = 8
    length_penalty: float = 1.3
    early_stopping: bool = True
    repetition_penalty: float = 1.2
    sample_top_p: float = 0.92
    sample_temperature: float = 0.75

    # MBR settings
    mbr_pool_cap: int = 32
    agreement_bonus: float = 0.03

    # Precision & optimization
    use_mixed_precision: bool = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True
    checkpoint_freq: int = 200

    seed: int = 42

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_paths = [self.model_a_path, self.model_b_path]

        if not torch.cuda.is_available():
            self.use_mixed_precision = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _cuda_bf16_supported()
        )

        assert self.num_beams >= self.num_beam_cands


cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print(f"Device: {cfg.device}")
print(f"BF16 AMP: {cfg.use_bf16_amp}")
print(f"Model A: {cfg.model_a_path}")
print(f"Model B: {cfg.model_b_path}")
print(f"Pool per model: beam×{cfg.num_beam_cands} + sample×{cfg.num_sample_cands}")

In [ ]:
# ==============================================================================
# 2. PREPROCESSING — From baseline: ASCII→diacritics, determinatives, gaps, fractions
# ==============================================================================

# --- ASCII to diacritics ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

# --- Fraction normalization ---
_ALLOWED_FRACS = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")

_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}
def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]

# --- Gap normalization ---
_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)

# --- Character cleanup ---
_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

# --- Determinative handling ---
_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_KUBABBAR_RE = re.compile(r"KÙ\.B\.")
_WS_RE = re.compile(r"\s+")


def preprocess_batch(texts: List[str]) -> List[str]:
    """Baseline-style batch preprocessing."""
    ser = pd.Series(texts).fillna("").astype(str)
    ser = ser.apply(_ascii_to_diacritics)

    # Uppercase determinatives unwrapped, lowercase → braces
    ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
    ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

    ser = _normalize_gaps_vec(ser)
    ser = ser.str.translate(_CHAR_TRANS)
    ser = ser.str.replace(_SUB_X, "", regex=False)
    ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
    ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
    ser = ser.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
    ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
    return ser.tolist()


# ==============================================================================
# 3. POST-PROCESSING — From baseline: grammar, months, commodities, shekels
# ==============================================================================
_PN_RE = re.compile(r"\bPN\b")

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])") 

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}
def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]

_SHEKEL_FIXES = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_SLASH_ALT_RE = re.compile(r'(?<!\d)\s*/\s*(?!\d)\S+')
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


def postprocess_batch(translations: List[str]) -> List[str]:
    """Baseline-style batch postprocessing."""
    s = pd.Series(translations).fillna("").astype(str)

    s = _normalize_gaps_vec(s)
    s = s.str.replace(_PN_RE, "<gap>", regex=True)
    s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

    for pat, repl in _SHEKEL_FIXES:
        s = s.str.replace(pat, repl, regex=True)

    s = s.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
    s = s.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)

    s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
    s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
    s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

    s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
    s = s.str.replace(_SLASH_ALT_RE, "", regex=True)

    # Remove curly quotes only
    s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

    s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
    s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

    # Strip forbidden chars but preserve <gap>
    s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
    s = s.str.translate(_FORBIDDEN_TRANS)
    s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

    # Remove repeated words / phrases
    s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
    for n in range(4, 1, -1):
        pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
        s = s.str.replace(pat, r"\1", regex=True)

    s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
    s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
    s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

    return s.tolist()


print("Pre/post-processing ready (baseline-enhanced).")

In [ ]:
# ==============================================================================
# 4. PROPER NOUN LEXICON + MBR SCORING (hand-written chrF++ matching sacrebleu)
#    No external dependencies — works offline on Kaggle
# ==============================================================================

def build_proper_noun_lexicon(csv_path: str) -> Dict[str, str]:
    """Load proper noun mapping from OA_Lexicon_eBL.csv."""
    try:
        df = pd.read_csv(csv_path, encoding='utf-8')
    except FileNotFoundError:
        print(f"Warning: Lexicon not found at {csv_path}, skipping proper noun scoring.")
        return {}

    target_types = ['PN', 'GN', 'DN', 'RN']
    entity_df = df[df['type'].isin(target_types)].copy()

    lexicon_map = {}
    for _, row in entity_df.iterrows():
        form = str(row['form']).strip()
        norm = str(row['norm']).strip()
        if pd.isna(form) or pd.isna(norm) or form == 'nan' or norm == 'nan':
            continue
        clean_form = re.sub(r'[\[\]\(\)\?\!]', '', form).lower()
        if clean_form:
            lexicon_map[clean_form] = norm

    print(f"Loaded {len(lexicon_map)} proper nouns into lexicon map.")
    return lexicon_map


# Load lexicon
PROPER_NOUN_MAP = build_proper_noun_lexicon(cfg.lexicon_csv_path)


# ---------- Hand-written chrF++ (matching sacrebleu's algorithm) ----------

def _ngram_counts(sequence, n: int) -> dict:
    """Count n-grams in a sequence (works for both chars and word lists)."""
    counts = {}
    for i in range(len(sequence) - n + 1):
        ng = sequence[i : i + n] if isinstance(sequence, str) else tuple(sequence[i : i + n])
        counts[ng] = counts.get(ng, 0) + 1
    return counts


def _fscore(precision: float, recall: float, beta: float = 2.0) -> float:
    if precision <= 0 or recall <= 0:
        return 0.0
    beta2 = beta * beta
    return (1 + beta2) * precision * recall / (beta2 * precision + recall)


def sentence_chrfpp(hyp: str, ref: str, char_order: int = 6,
                    word_order: int = 2, beta: float = 2.0) -> float:
    """
    ChrF++ matching sacrebleu's algorithm:
    1. For each n-gram order, compute F-score separately
    2. Average char F-scores → avg_char_f
    3. Average word F-scores → avg_word_f
    4. Final = weighted average by number of orders
    """
    hyp, ref = hyp.strip(), ref.strip()
    if not hyp or not ref:
        return 0.0

    # --- Character n-gram F-scores ---
    char_fscores = []
    for n in range(1, char_order + 1):
        hyp_ng = _ngram_counts(hyp, n)
        ref_ng = _ngram_counts(ref, n)
        if not hyp_ng or not ref_ng:
            char_fscores.append(0.0)
            continue
        overlap = sum(min(c, ref_ng.get(ng, 0)) for ng, c in hyp_ng.items())
        p = overlap / sum(hyp_ng.values())
        r = overlap / sum(ref_ng.values())
        char_fscores.append(_fscore(p, r, beta))

    # --- Word n-gram F-scores ---
    hyp_w, ref_w = hyp.split(), ref.split()
    word_fscores = []
    for n in range(1, word_order + 1):
        hyp_ng = _ngram_counts(hyp_w, n)
        ref_ng = _ngram_counts(ref_w, n)
        if not hyp_ng or not ref_ng:
            word_fscores.append(0.0)
            continue
        overlap = sum(min(c, ref_ng.get(ng, 0)) for ng, c in hyp_ng.items())
        p = overlap / sum(hyp_ng.values())
        r = overlap / sum(ref_ng.values())
        word_fscores.append(_fscore(p, r, beta))

    # --- Weighted average (sacrebleu style) ---
    total_orders = char_order + word_order
    if total_orders == 0:
        return 0.0

    avg_char_f = sum(char_fscores) / max(1, len(char_fscores))
    avg_word_f = sum(word_fscores) / max(1, len(word_fscores))

    score = (char_order * avg_char_f + word_order * avg_word_f) / total_orders
    return score * 100.0  # scale to 0-100


class MBRSelector:
    """MBR consensus using chrF++ with proper noun fidelity bonus."""

    def __init__(self, pool_cap: int = 32, agreement_bonus: float = 0.03,
                 w_chrf: float = 0.8, w_fidelity: float = 0.2):
        self.pool_cap = pool_cap
        self.agreement_bonus = agreement_bonus
        self.lexicon_map = PROPER_NOUN_MAP
        self.w_chrf = w_chrf
        self.w_fidelity = w_fidelity

    def _chrfpp(self, a: str, b: str) -> float:
        return sentence_chrfpp(a, b)

    def _lexical_fidelity_score(self, source_text: str, candidate: str) -> float:
        """Score how well the candidate preserves proper nouns from source."""
        if not source_text or not candidate:
            return 0.0

        clean_source = re.sub(r'[^\w\-\s]', '', source_text.lower())
        source_tokens = clean_source.split()

        expected_entities = []
        for token in source_tokens:
            if token in self.lexicon_map:
                expected_entities.append(self.lexicon_map[token].lower())

        # No proper nouns in source → full score (don't interfere with consensus)
        if not expected_entities:
            return 100.0

        cand_lower = candidate.lower()
        match_count = sum(1 for entity in expected_entities if entity in cand_lower)
        return (match_count / len(expected_entities)) * 100.0

    @staticmethod
    def _dedup_with_counts(xs: List[str]) -> Tuple[List[str], Dict[str, int]]:
        counts, order = {}, []
        for x in xs:
            x = str(x or "").strip()
            if not x:
                continue
            if x not in counts:
                order.append(x)
                counts[x] = 0
            counts[x] += 1
        return order, counts

    def pick(self, source_text: str, candidates: List[str]) -> str:
        cands, counts = self._dedup_with_counts(candidates)

        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        best_i, best_s = 0, -1e9
        for i in range(n):
            # 1. chrF++ consensus score
            consensus = sum(self._chrfpp(cands[i], cands[j]) for j in range(n) if j != i)
            consensus /= max(1, n - 1)

            # 2. Proper noun fidelity score
            fidelity = self._lexical_fidelity_score(source_text, cands[i])

            # 3. Agreement bonus for duplicates (multiple models agree)
            dup_bonus = self.agreement_bonus * max(0, counts.get(cands[i], 1) - 1)

            # 4. Weighted total
            total = (self.w_chrf * consensus) + (self.w_fidelity * fidelity) + dup_bonus

            if total > best_s:
                best_s, best_i = total, i

        return cands[best_i]


print("MBR scoring ready (chrF++ matching sacrebleu algorithm, no external deps).")

In [ ]:
# ==============================================================================
# 5. CANDIDATE GENERATION — BF16 autocast + OOM protection + repetition_penalty
# ==============================================================================

def generate_candidates(model, tokenizer, texts: list, cfg: Config) -> list:
    """Generate beam + sampling candidates with BF16 autocast and OOM safety."""
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True,
        max_length=cfg.max_input_length,
    ).to(cfg.device)

    B = len(texts)
    ctx = _bf16_ctx(cfg.device, cfg.use_bf16_amp)

    try:
        with ctx:
            # Beam search candidates
            nb = max(cfg.num_beams, cfg.num_beam_cands)
            beam_out = model.generate(
                **inputs,
                do_sample=False,
                num_beams=nb,
                num_return_sequences=cfg.num_beam_cands,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,
                use_cache=True,
            )
            beam_texts = tokenizer.batch_decode(beam_out, skip_special_tokens=True)

            # Sampling candidates (diverse, for MBR pool)
            sample_texts = []
            if cfg.num_sample_cands > 0:
                sample_out = model.generate(
                    **inputs,
                    do_sample=True,
                    num_beams=1,
                    top_p=cfg.sample_top_p,
                    temperature=cfg.sample_temperature,
                    num_return_sequences=cfg.num_sample_cands,
                    max_new_tokens=cfg.max_new_tokens,
                    repetition_penalty=cfg.repetition_penalty,
                    use_cache=True,
                )
                sample_texts = tokenizer.batch_decode(sample_out, skip_special_tokens=True)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"  [OOM] Skipping batch of {B} — clearing cache")
            torch.cuda.empty_cache()
            return [[] for _ in range(B)]
        raise

    # Group by input
    rb, rs = cfg.num_beam_cands, cfg.num_sample_cands
    pools = []
    for i in range(B):
        p = list(beam_texts[i * rb : (i + 1) * rb])
        if rs > 0 and sample_texts:
            p.extend(sample_texts[i * rs : (i + 1) * rs])
        pools.append(p)
    return pools


# ==============================================================================
# 6. BUCKET BATCHING (from baseline) — reduces padding waste
# ==============================================================================

class BucketBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, num_buckets, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle

        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsize = max(1, len(sorted_idx) // max(1, num_buckets))
        self.buckets = [
            sorted_idx[i * bsize : None if i == num_buckets - 1 else (i + 1) * bsize]
            for i in range(num_buckets)
        ]

        for i, b in enumerate(self.buckets):
            if b:
                bl = [lengths[x] for x in b]
                print(f"  Bucket {i}: {len(b)} samples, len [{min(bl)}, {max(bl)}]")

    def __iter__(self):
        for bucket in self.buckets:
            b = list(bucket)
            if self.shuffle:
                random.shuffle(b)
            for i in range(0, len(b), self.batch_size):
                yield b[i:i + self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(b) / self.batch_size) for b in self.buckets)


print("Candidate generation + bucket batching ready.")

In [ ]:
# ==============================================================================
# 7. MAIN PIPELINE — Multi-model ensemble + MBR with all baseline enhancements
# ==============================================================================
from pathlib import Path

# Load test data
test_df = pd.read_csv(cfg.test_data_path)
raw_transliterations = test_df["transliteration"].tolist()
test_sources = preprocess_batch(raw_transliterations)
test_inputs = ["translate Akkadian to English: " + s for s in test_sources]
test_ids = test_df["id"].tolist()

print(f"Test samples: {len(test_inputs)}")
print(f"Models to ensemble: {len(cfg.model_paths)}")
print(f"Candidates per model per sample: {cfg.num_beam_cands} beam + {cfg.num_sample_cands} sampling")
print(f"Total candidate pool per sample: ~{len(cfg.model_paths) * (cfg.num_beam_cands + cfg.num_sample_cands)}")
print(f"BF16 AMP: {cfg.use_bf16_amp}")

# Build bucket batching indices
input_lengths = [len(t.split()) for t in test_inputs]
if cfg.use_bucket_batching:
    print("\nBucket batching:")
    bucket_sampler = BucketBatchSampler(input_lengths, cfg.batch_size, cfg.num_buckets)
    batches = list(bucket_sampler)
else:
    batches = [list(range(i, min(i + cfg.batch_size, len(test_inputs))))
               for i in range(0, len(test_inputs), cfg.batch_size)]

# Collect candidate pools across all models
all_pools = {i: [] for i in range(len(test_inputs))}

for model_idx, model_path in enumerate(cfg.model_paths):
    label = os.path.basename(model_path) or f"model_{model_idx}"
    print(f"\n--- Loading {label} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)
    model.to(cfg.device).eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {n_params:,} parameters")
    if cfg.device.type == "cuda":
        used = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU mem used: {used:.2f} GB")

    with torch.inference_mode():
        for batch_indices in tqdm(batches, desc=f"{label}"):
            batch_texts = [test_inputs[i] for i in batch_indices]
            pools = generate_candidates(model, tokenizer, batch_texts, cfg)
            for j, pool in enumerate(pools):
                all_pools[batch_indices[j]].extend(pool)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Free memory before loading next model
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
        print(f"  [{label}] Unloaded. GPU free: {free:.2f} GB")

# MBR consensus selection + post-processing
print("\n--- MBR Consensus Selection (chrF++ + proper noun fidelity) ---")
mbr = MBRSelector(
    pool_cap=cfg.mbr_pool_cap,
    agreement_bonus=cfg.agreement_bonus,
)

final_translations = []
for i in tqdm(range(len(test_inputs)), desc="MBR"):
    # Post-process all candidates
    cleaned = postprocess_batch(all_pools[i]) if all_pools[i] else []

    # MBR pick with source text for proper noun scoring
    source_text = raw_transliterations[i]
    best = mbr.pick(source_text, cleaned)
    final_translations.append(best if best.strip() else "The tablet is too damaged to translate.")

    # Periodic checkpoint
    if (i + 1) % cfg.checkpoint_freq == 0:
        ckpt = Path(cfg.output_dir) / f"checkpoint_{i+1}.csv"
        pd.DataFrame({
            "id": test_ids[:i+1],
            "translation": final_translations
        }).to_csv(ckpt, index=False)
        print(f"  Checkpoint: {i+1} rows → {ckpt}")

# Write submission
submission = pd.DataFrame({"id": test_df["id"], "translation": final_translations})
submission.to_csv(Path(cfg.output_dir) / "submission.csv", index=False)

# Validation summary
empty = submission["translation"].str.strip().eq("").sum()
lens = submission["translation"].str.len()
print(f"\n{'='*60}")
print(f"submission.csv written! ({len(submission)} rows)")
print(f"Empty: {empty} ({100 * empty / max(1, len(submission)):.2f}%)")
print(f"Len mean: {lens.mean():.1f}  median: {lens.median():.1f}  min: {lens.min()}  max: {lens.max()}")
print(f"{'='*60}")

print("\nSample predictions:")
for idx in [0, len(submission)//4, len(submission)//2, 3*len(submission)//4, len(submission)-1]:
    row = submission.iloc[idx]
    print(f"  ID {row['id']}: {str(row['translation'])[:120]}")

submission